# Imports

In [ ]:
import pandas as pd
from importlib import reload
import plotly.express as px
from pathlib import Path

In [ ]:
figure_path = Path("./figures")
figure_tag = "07_scaffold_split_analysis"

In [ ]:
figure_path.mkdir(parents=True, exist_ok=True)

In [ ]:
def get_svg(fig_name) -> Path:
    return figure_path / f"{figure_tag}_{fig_name}.svg"

def get_png(fig_name) -> Path:
    return figure_path / f"{figure_tag}_{fig_name}.png"

def save_both(fig, fig_name=""):
    fig.write_image(get_svg(fig_name))
    fig.write_image(get_png(fig_name))
def save_both_plt(fig_name=""):
    plt.savefig(get_svg(fig_name))
    plt.savefig(get_png(fig_name))

In [ ]:
raw_df = pd.read_csv("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/combined_results.csv")

In [ ]:
raw_df.groupby(["Split_Option_1", "Score"]).nunique()

In [ ]:
raw_df["Query_Scaffold_ID_Subset_1"] = raw_df["Query_Scaffold_ID_Subset_1"].astype(str).apply(lambda x: x.replace("[", "").replace("]", "") if "[" in x else x)
raw_df["Query_Scaffold_ID_Subset_1_int"] = raw_df["Query_Scaffold_ID_Subset_1"].astype(float)
raw_df["Reference_Scaffold_ID_Subset_1"] = raw_df["Reference_Scaffold_ID_Subset_1"].astype(str).apply(lambda x: x.replace("[", "").replace("]", "") if "[" in x else x)
raw_df["Reference_Scaffold_ID_Subset_1_int"] = raw_df["Reference_Scaffold_ID_Subset_1"].astype(float)

In [ ]:
raw_df["Error_Upper"] = raw_df["CI_Upper"] - raw_df["Fraction"]
raw_df["Error_Lower"] = raw_df["Fraction"] - raw_df["CI_Lower"]

# NOT X TO X

In [ ]:
raw_df.sort_values(["Reference_Scaffold_ID_Subset_1_int", "Split_Option_1", "Score", "N_Per_Split"], ascending=[True, True, True, True], inplace=True)
fig = px.line(raw_df[raw_df["Split_Option_1"] == "ScaffoldSplitOptions.NOT_X_TO_X"], 
              x="N_Per_Split", 
              y="Fraction",  
              color="Reference_Scaffold_ID_Subset_1",
              facet_col='Score',
              line_dash='Split',
              template='simple_white',
              log_x=True,
              height=600,
              width=1200)
fig.show()
save_both(fig, "not_x_to_x")

## what if we skip the n_per_split, just using the final (max) result?

In [ ]:
not_x_to_x = raw_df[raw_df["Split_Option_1"] == "ScaffoldSplitOptions.NOT_X_TO_X"]
not_x_to_x_final = not_x_to_x.sort_values(["Reference_Scaffold_ID_Subset_1_int", "N_Per_Split"]).groupby(["Reference_Scaffold_ID_Subset_1", "Score", "Split"]).tail(1)

In [ ]:
fig = px.bar(not_x_to_x_final,
             x="Reference_Scaffold_ID_Subset_1",
             y="Fraction",
             # color="Split_Score",  # Use combined column for coloring
             color="Split",
             pattern_shape="Score",
             barmode="group",
             template='simple_white',
             height=300,
             width=1200)

# Update layout
fig.update_layout(
    xaxis_title="Reference Scaffold",
    yaxis_title="Fraction",
    legend_title="Split - Score"
)

fig.show()
save_both(fig, "not_x_to_x_final_bar")

### try with only one split

In [ ]:
# the split shouldn't matter here
not_x_to_x_final_one_split= not_x_to_x_final[not_x_to_x_final["Split"] == "DateSplit"]

In [ ]:
not_x_to_x_final_one_split

In [ ]:
fig = px.scatter(not_x_to_x_final_one_split,
             x="Reference_Scaffold_ID_Subset_1",
             y="Fraction",
             # color="Split_Score",  # Use combined column for coloring
             color="Score",
             error_y= "Error_Upper",
             error_y_minus="Error_Lower",
             template='simple_white',)
large_font=24
small_font=16
tick_font=16
fig.update_layout(
    # Figure dimensions
    height=600,
    width=600,
    template="simple_white",
    margin=dict(l=0, r=20, t=20, b=0),
    title=dict(text="<b> Docking all other scaffolds to each scaffold </b>", font=dict(size=large_font, family='Helvetica'), x=1, y=0.975, xanchor='right', yanchor='top'),
    
    # Font settings
    font_family="Helvetica",
    
    # Legend Settings
    legend=dict(
        title_text="<b> Score </b>",
        # orientation="h",     # horizontal legend
        yanchor="bottom",   # anchor point
        y=0,            # position above the plot
        xanchor="right",    # anchor point
        x=1.1,                # position at the right
        font=dict(size=small_font, family='Helvetica'), title=dict(font=dict(size=large_font, family='Helvetica'))
    ),
    
    # X-axis settings
    xaxis=dict(
        title_text="<b> Reference Scaffold </b>",
        title_font=dict(size=24),
        color="black",
        tickfont=dict(size=16, family="Helvetica", color="black"),
        tickmode="auto",  # Can be "auto", "linear", "array"
        # nticks=10,        # Approximate number of ticks when using "auto" mode
        tickangle=0,      # Rotation angle of tick labels
        ticklen=8,        # Length of tick marks
        tickwidth=2,      # Width of tick marks
        showgrid=False,    # Show grid lines
        gridcolor="lightgray",  # Color of grid lines
        gridwidth=1,      # Width of grid lines
    ),
    
    # Y-axis settings
    yaxis=dict(
        title_text="<b> Fraction of Ligands Posed <br> <2 Å from Crystal Pose </b>",
        title_font=dict(size=large_font),
        color="black",
        tickfont=dict(size=tick_font, family="Helvetica", color="black"),
        tickmode="auto",
        nticks=8,
        tickformat=".1f",  # Format for tick labels (1 decimal place)
        ticklen=8,
        tickwidth=2,
        showgrid=False,
        gridcolor="lightgray",
        gridwidth=1,
        range=(0,1),  # Commented out as in original
    ),
)
fig.show()
save_both(fig, "not_x_to_x_final_one_split_scatter")

# X TO NOT X

In [ ]:
x_to_not_x = raw_df[raw_df["Split_Option_1"] == "ScaffoldSplitOptions.X_TO_NOT_X"]
x_to_not_x_final = x_to_not_x.sort_values(["Query_Scaffold_ID_Subset_1_int", "N_Per_Split"]).groupby(["Query_Scaffold_ID_Subset_1", "Score", "Split"]).tail(1)
x_to_not_x_final_one_split = x_to_not_x_final[x_to_not_x_final["Split"] == "DateSplit"]

In [ ]:
fig = px.scatter(x_to_not_x_final_one_split,
             x="Query_Scaffold_ID_Subset_1",
             y="Fraction",
             # color="Split_Score",  # Use combined column for coloring
             color="Score",
             error_y= "Error_Upper",
             error_y_minus="Error_Lower",
             template='simple_white',)
large_font=24
small_font=16
tick_font=16
fig.update_layout(
    # Figure dimensions
    height=600,
    width=600,
    template="simple_white",
    margin=dict(l=0, r=20, t=20, b=0),
    title=dict(text="<b> Docking each scaffold to all other scaffolds </b>", font=dict(size=large_font, family='Helvetica'), x=1, y=0.975, xanchor='right', yanchor='top'),
    
    # Font settings
    font_family="Helvetica",
    
    # Legend Settings
    legend=dict(
        title_text="<b> Score </b>",
        # orientation="h",     # horizontal legend
        yanchor="bottom",   # anchor point
        y=0,            # position above the plot
        xanchor="right",    # anchor point
        x=1.1,                # position at the right
        font=dict(size=small_font, family='Helvetica'), title=dict(font=dict(size=large_font, family='Helvetica'))
    ),
    
    # X-axis settings
    xaxis=dict(
        title_text="<b> Query Scaffold </b>",
        title_font=dict(size=24),
        color="black",
        tickfont=dict(size=16, family="Helvetica", color="black"),
        tickmode="auto",  # Can be "auto", "linear", "array"
        # nticks=10,        # Approximate number of ticks when using "auto" mode
        tickangle=0,      # Rotation angle of tick labels
        ticklen=8,        # Length of tick marks
        tickwidth=2,      # Width of tick marks
        showgrid=False,    # Show grid lines
        gridcolor="lightgray",  # Color of grid lines
        gridwidth=1,      # Width of grid lines
    ),
    
    # Y-axis settings
    yaxis=dict(
        title_text="<b> Fraction of Ligands Posed <br> <2 Å from Crystal Pose </b>",
        title_font=dict(size=large_font),
        color="black",
        tickfont=dict(size=tick_font, family="Helvetica", color="black"),
        tickmode="auto",
        nticks=8,
        tickformat=".1f",  # Format for tick labels (1 decimal place)
        ticklen=8,
        tickwidth=2,
        showgrid=False,
        gridcolor="lightgray",
        gridwidth=1,
        range=(0,1.1),  # Commented out as in original
    ),
)
fig.show()
save_both(fig, "x_to_not_x_final_one_split_scatter")

In [ ]:
x_to_not_x_final_one_split

# X TO Y

In [ ]:
scaffold_data = pd.read_csv("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/scaffold_split_x_to_y/generic_cluster_labels.csv")

In [ ]:
lig_to_scaffold_dict = {data["compound_name"]: data["cluster_id"] for data in scaffold_data.to_dict(orient='records')}

In [ ]:
import json
with open("/Users/alexpayne/Scientific_Projects/asapdiscovery-sars-retrospective/data/cmpd_date_dict/structure_to_cmpd_dict.json", 'r') as f:
    ligand_data = json.load(f)

In [ ]:
from collections import defaultdict
p_series_scaffolds = defaultdict(list)
x_series_scaffolds = defaultdict(list)
series_label = [{"compound_id": ligand_id, "IS_PSERIES": True if "-P" in dataset else False, "IS_XSERIES": True if "-x" in dataset else False} 
                                          for dataset, ligand_id in ligand_data.items()]
for data in scaffold_data.to_dict(orient='records'):
    compound_name = data['compound_name']
    for series_data in series_label:
        if compound_name == series_data['compound_id']:
            if series_data['IS_PSERIES']:
                p_series_scaffolds[data["cluster_id"]].append(compound_name)
            if series_data['IS_XSERIES']:
                x_series_scaffolds[data["cluster_id"]].append(compound_name)
p_series_scaffolds = {cluster_id: set(compounds) for cluster_id, compounds in p_series_scaffolds.items()}
x_series_scaffolds = {cluster_id: set(compounds) for cluster_id, compounds in x_series_scaffolds.items()}

In [ ]:
p_series_scaffold_counts = {cluster_id: len(compounds) for cluster_id, compounds in p_series_scaffolds.items()}
x_series_scaffold_counts = {cluster_id: len(compounds) for cluster_id, compounds in x_series_scaffolds.items()}

In [ ]:
x_to_y = pd.read_csv("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/scaffold_split_x_to_y/combined_results.csv", index_col=0)

In [ ]:
raw_df = x_to_y
raw_df["Query_Scaffold_ID_Subset_1"] = raw_df["Query_Scaffold_ID_Subset_1"].astype(str).apply(lambda x: x.replace("[", "").replace("]", "") if "[" in x else x)
raw_df["qint"] = raw_df["Query_Scaffold_ID_Subset_1"].astype(float)
raw_df["Reference_Scaffold_ID_Subset_1"] = raw_df["Reference_Scaffold_ID_Subset_1"].astype(str).apply(lambda x: x.replace("[", "").replace("]", "") if "[" in x else x)
raw_df["rint"] = raw_df["Reference_Scaffold_ID_Subset_1"].astype(float)

In [ ]:
raw_df["Error_Upper"] = raw_df["CI_Upper"] - raw_df["Fraction"]
raw_df["Error_Lower"] = raw_df["Fraction"] - raw_df["CI_Lower"]

In [ ]:
x_to_y = raw_df

In [ ]:
x_to_y.sort_values(["rint", "qint", "N_Per_Split"], inplace=True)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Determine unique values for facets
unique_scores = x_to_y['Score'].unique()
unique_splits = x_to_y['Split'].unique()

# Create subplots
fig = make_subplots(
    rows=len(unique_splits),
    cols=len(unique_scores),
    subplot_titles=[f'Score: {score}' for score in unique_scores] * len(unique_splits),
    x_title="N_Per_Split",
    y_title="Fraction"
)

# Create color and dash styles
base_colors = px.colors.qualitative.Dark24[:14]
line_dashes = ['solid', 'dot', 'dash', 'longdash', 'dashdot', 'longdashdot'] * 3

# Add traces for each combination
for split_idx, split in enumerate(unique_splits, 1):
    for score_idx, score in enumerate(unique_scores, 1):
        split_score_data = x_to_y[
            (x_to_y['Split'] == split) & 
            (x_to_y['Score'] == score)
        ]
        
        ref_ids = split_score_data['Reference_Scaffold_ID_Subset_1'].unique()
        query_ids = split_score_data['Query_Scaffold_ID_Subset_1'].unique()
        
        for ref_id in ref_ids:
            for query_id in query_ids:
                data = split_score_data[
                    (split_score_data['Reference_Scaffold_ID_Subset_1'] == ref_id) &
                    (split_score_data['Query_Scaffold_ID_Subset_1'] == query_id)
                ]
                
                # Show legend for all references with first query, and all queries with first reference
                show_ref = split_idx == 1 and score_idx == 1 and query_id == query_ids[0]
                show_query = split_idx == 1 and score_idx == 1 and ref_id == ref_ids[0]

                fig.add_trace(
                    go.Scatter(
                        x=data['N_Per_Split'],
                        y=data['Fraction'],
                        name=f'{ref_id} - {query_id}',
                        line=dict(
                            color=base_colors[int(ref_id) % len(base_colors)],
                            dash=line_dashes[int(query_id) % len(line_dashes)],
                            width=2
                        ),
                        legendgroup='Reference' if show_ref else 'Query',
                        legendgrouptitle_text='Reference Scaffolds' if show_ref else 'Query Scaffolds',
                        showlegend=show_ref or show_query
                    ),
                    row=split_idx,
                    col=score_idx
                )
                if show_ref and show_query:
                    fig.add_trace(
                        go.Scatter(
                            x=data['N_Per_Split'],
                            y=data['Fraction'],
                            name=f'{ref_id} - {query_id}',
                            line=dict(
                                color=base_colors[int(ref_id) % len(base_colors)],
                                dash=line_dashes[int(query_id) % len(line_dashes)],
                                width=2
                            ),
                            legendgroup='Query',
                            legendgrouptitle_text='Query Scaffolds',
                            showlegend=show_ref or show_query
                        ),
                        row=split_idx,
                        col=score_idx
                    )
                

# Update layout
fig.update_layout(
    template='simple_white',
    height=800,
    width=1200,
    legend=dict(
        groupclick="toggleitem",
        tracegroupgap=20
    )
)

# Update all subplot x-axes to log scale
for i in range(1, len(unique_splits) * len(unique_scores) + 1):
    fig.update_xaxes(type='log', row=(i-1)//len(unique_scores) + 1, col=((i-1)%len(unique_scores)) + 1)

In [ ]:
fig.show()

## try with seaborn

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
single_category = x_to_y[(x_to_y["Score"]=="RMSD")&(x_to_y["Split"]=="RandomSplit")]
single_category["ref_int"] = single_category["Reference_Scaffold_ID_Subset_1"].astype(int)
single_category["query_int"] = single_category["Query_Scaffold_ID_Subset_1"].astype(int)
single_category.sort_values(["ref_int","query_int", "N_Per_Split"], inplace=True)

In [ ]:
from matplotlib.ticker import ScalarFormatter
import numpy as np

# Set the figure size before creating the plot
plt.figure(figsize=(12, 8))

sns.lineplot(data=single_category, x='N_Per_Split',
             y='Fraction',
             hue='Reference_Scaffold_ID_Subset_1',
             style='Query_Scaffold_ID_Subset_1')

# Add labels and title
plt.xlabel('Category')
plt.ylabel('Value')
plt.title('Bar Plot with Different Color Sequences for Categories')

# Set x-axis to log scale but show raw numbers
plt.xscale('log')
plt.gca().xaxis.set_major_formatter(ScalarFormatter())

# Set custom tick locations
custom_ticks = [1, 5, 10, 20, 50, 100, 200, single_category["N_Per_Split"].nunique()]
plt.xticks(custom_ticks, custom_ticks)

# Move legend outside to the right
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Adjust layout to prevent legend cutoff
plt.tight_layout()

plt.show()

# Make heatmap

In [ ]:
heatmap = x_to_y.sort_values(["N_Per_Split"], ascending=False).groupby(["Query_Scaffold_ID_Subset_1", "Reference_Scaffold_ID_Subset_1", "Score", "Split"]).head(1)

In [ ]:
heatmap_rmsd_random = heatmap[(heatmap.Split == "RandomSplit")&(heatmap.Score == "RMSD")]

In [ ]:
heatmap_rmsd_random = heatmap_rmsd_random.sort_values(['rint', 'qint'])

In [ ]:
sns.scatterplot(heatmap_rmsd_random, 
                x="rint", 
                y="qint", 
                hue="Total", 
                size='Fraction',
                palette=sns.color_palette("crest_r", as_cmap=True))

# Move legend outside to the right
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Adjust layout to prevent legend cutoff
plt.tight_layout()

plt.show()

In [ ]:
raw_df["Query_Scaffold_ID_Subset_1"] = raw_df["Query_Scaffold_ID_Subset_1"].astype(str).apply(lambda x: x.replace("[", "").replace("]", "") if "[" in x else x)
raw_df["Query_Scaffold_ID_Subset_1_int"] = raw_df["Query_Scaffold_ID_Subset_1"].astype(float)
raw_df["Reference_Scaffold_ID_Subset_1"] = raw_df["Reference_Scaffold_ID_Subset_1"].astype(str).apply(lambda x: x.replace("[", "").replace("]", "") if "[" in x else x)
raw_df["Reference_Scaffold_ID_Subset_1_int"] = raw_df["Reference_Scaffold_ID_Subset_1"].astype(float)

In [ ]:
raw_df["Error_Upper"] = raw_df["CI_Upper"] - raw_df["Fraction"]
raw_df["Error_Lower"] = raw_df["Fraction"] - raw_df["CI_Lower"]

In [ ]:
ref_totals = heatmap_rmdsd_random.sort_values(["Reference_Scaffold_ID_Subset_1_int"]).groupby("Reference_Scaffold_ID_Subset_1_int").nunique()

In [ ]:
pivot_total = heatmap_rmsd_random.pivot(index='qint',
                                       columns='rint',
                                       values='Total')

In [ ]:
pivot_total

In [ ]:
pivot_total_r = heatmap_rmsd_random.pivot(index='rint',
                                       columns='qint',
                                       values='Total')

In [ ]:
pivot_total_r

In [ ]:
heatmap_rmsd_random[heatmap_rmsd_random['rint'] == 3]

In [ ]:
# Sort the unique values numerically

ref_order = sorted(heatmap_rmsd_random['rint'].unique())
query_order = sorted(heatmap_rmsd_random['qint'].unique(), key=int)

# Create DataFrames
pivot_total = heatmap_rmsd_random.pivot(index='qint',
                                       columns='rint',
                                       values='Total')
pivot_fraction = heatmap_rmsd_random.pivot(index='qint',
                                         columns='rint',
                                         values='Fraction')



# Create a DataFrame for paired totals
paired_totals = pivot_total.copy().astype(str)
for i in pivot_total.index:
    for j in pivot_total.coluns:
        if i in pivot_total.columns and j in pivot_total.index:
            paired_totals.loc[i, j] = f"{p_series_scaffold_counts.get(int(i), 0)}/{x_series_scaffold_counts.get(int(j), 0)}"

plt.figure(figsize=(8, 6))
heatmap = sns.heatmap(data=pivot_fraction,
                      cmap="flare",
                      annot=paired_totals,
                      fmt='',
                      xticklabels=ref_order,
                      yticklabels=query_order)

# Add colorbar label
heatmap.collections[0].colorbar.set_label('Fraction')

# Rotate axis labels for better readability
plt.xticks(rotation=0)
plt.yticks(rotation=0)

# Invert y-axis to put 0 at bottom
plt.gca().invert_yaxis()

# Add labels
plt.xlabel('Reference Interval')
plt.ylabel('Query Interval')

plt.tight_layout()
plt.show()

In [ ]:
heatmap_rmsd_random[heatmap_rmsd_random["qint"] == 13].groupby("rint").max()

In [ ]:
ceviche = pd.read_csv("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/20250212_p_to_x_posit/20250311_combined_results")

In [ ]:
ceviche.columns

In [ ]:
ceviche[ceviche["cluster_id_Reference"] == 0].nunique()["Reference_Ligand"]

In [ ]:
set1 = set(ceviche[ceviche["cluster_id_Reference"] == 7]["Reference_Ligand"].unique())

In [ ]:
heatmap_rmsd_random[heatmap_rmsd_random['rint'] == 7]

In [ ]:
set2 = 

In [ ]:
ceviche[ceviche["cluster_id"] == 0].nunique()["Query_Ligand"]

In [ ]:
# Sort the unique values numerically
ref_order = sorted(heatmap_rmsd_random['Reference_Scaffold_ID_Subset_1'].unique(), key=int)
query_order = sorted(heatmap_rmsd_random['Query_Scaffold_ID_Subset_1'].unique(), key=int)

# Create DataFrames
pivot_total = heatmap_rmsd_random.pivot(index='qint',
                                       columns='rint',
                                       values='Total')
pivot_fraction = heatmap_rmsd_random.pivot(index='qint',
                                         columns='rint',
                                         values='Fraction')

# Pre-compute unique counts
query_counts = ceviche.groupby('cluster_id')['Query_Ligand'].nunique()
ref_counts = ceviche.groupby('cluster_id_Reference')['Reference_Ligand'].nunique()

# Create annotations with blank for zeros
# annotations = pivot_fraction.applymap(lambda x: '' if x == 0 else f'{x:.1f}')

# Create annotations with blank only when both ref and query counts are 0
def format_value(x, i, j):
    if x == 0 and query_counts.get(int(i), 0) == 0 or ref_counts.get(int(j), 0) == 0:
        return ''
    return f'{x:.1f}'

annotations = pd.DataFrame(
    [[format_value(pivot_fraction.iloc[i, j], pivot_fraction.index[i], pivot_fraction.columns[j]) 
      for j in range(len(pivot_fraction.columns))]
     for i in range(len(pivot_fraction.index))],
    index=pivot_fraction.index,
    columns=pivot_fraction.columns
)

plt.figure(figsize=(8, 6))
sns.set(font='Helvetica')
heatmap = sns.heatmap(data=pivot_fraction,
                      cmap="flare",
                      annot=annotations,
                      fmt='',
                      xticklabels=[f"$\\bf{cluster_id}$\n({ref_counts.get(int(cluster_id), 0)})" for cluster_id in ref_order],
                      yticklabels=[f"$\\bf{cluster_id}$ ({query_counts.get(int(cluster_id), 0)})" for cluster_id in query_order])

# Add colorbar label
heatmap.collections[0].colorbar.set_label('Fraction')

# Rotate axis labels for better readability
plt.xticks(rotation=0)
plt.yticks(rotation=0)

# Invert y-axis to put 0 at bottom
plt.gca().invert_yaxis()

# Add labels
plt.xlabel('$\\bf{Reference\ Scaffold\ ID}$\n(# Reference Structures with Scaffold)')
plt.ylabel('$\\bf{Query\ Scaffold\ ID}$ (# Query Ligands with Scaffold)')

plt.tight_layout()

# Save the plot as an SVG file
fig_name = "x_to_y_heatmap"
plt.savefig(get_svg(fig_name))
plt.savefig(get_png(fig_name))